# Deequ in PySpark: Unit Tests for Data

Deequ is a library built on Apache Spark for defining *"unit tests for data"* — automated, declarative checks that measure data quality at scale. Just as you write unit tests so you don't ship broken code, you write Deequ checks so you don't ship (or train on) broken data.

This notebook focuses on the **core workflow**: defining constraints and running them through the `VerificationSuite`. We finish with a short teaser of automatic constraint suggestion.

**The big idea to keep in mind:** a check that *passes* is boring; the value of Deequ is that it *fails loudly and tells you exactly why* before bad data reaches your model or dashboard.

---
### Running this notebook in Google Colab

Colab gives you a clean Linux VM with a JVM already present (currently **Java 17**) but **no Spark**, so the first section installs Spark/PyDeequ and points Spark at the Java that's already there. Just run the cells top to bottom.

- Section 0 installs `pyspark` + `pydeequ`, **auto-detects** `JAVA_HOME`, and sets `SPARK_VERSION` — about a minute on first run.
- Spark 3.5 runs fine on Java 17, so we do **not** install a separate JDK. We detect whatever Java the runtime ships so the path can't drift if Google updates the image.
- You need an internet connection (Colab has one) so Spark can pull the Deequ JAR from Maven on first launch.
- Everything resets when the Colab runtime recycles, so if you come back later, just re-run Section 0.


## 0. Setup (Colab-specific)

Colab has Spark missing but a JVM already present, so here we:

1. Install matching **PySpark + PyDeequ** (and remove a preinstalled package that conflicts on the pyspark version).
2. **Auto-detect** `JAVA_HOME` from the Java the runtime actually ships, instead of hardcoding a path that could be wrong after a Colab image update.
3. Set `SPARK_VERSION` **before** importing pydeequ so it selects the matching Deequ JAR.

Run this cell first — it takes about a minute.

In [ ]:
# Colab preinstalls `dataproc-spark-connect`, which wants pyspark ~=4.0 and will
# print a scary red dependency-conflict ERROR. We don't use it, so remove it to
# keep the output clean.

!pip uninstall -y dataproc-spark-connect -q 2>/dev/null || true

# Install matching PySpark + PyDeequ. We pin PySpark 3.5.x so it lines up
# with the Deequ 2.0.x-spark-3.5 JAR that pydeequ will request.

%pip install -q "pyspark==3.5.1" pydeequ

**Auto-detect Java version**

In [ ]:
import os, subprocess, pathlib

# Resolve the real java binary (follow symlinks), then strip the trailing /bin/java
java_path = subprocess.check_output(["which", "java"]).decode().strip()
java_real = subprocess.check_output(["readlink", "-f", java_path]).decode().strip()
java_home = str(pathlib.Path(java_real).parent.parent)   # .../bin/java -> .../

os.environ["JAVA_HOME"] = java_home
os.environ["SPARK_VERSION"] = "3.5"      # must match the pyspark major.minor above

# Safety net for Java 17: Spark occasionally needs these module flags when launched
# directly (not via spark-submit). Harmless on Java 8/11. Uncomment if you hit an
# 'InaccessibleObjectException' or 'cannot access class sun.nio.ch...' on startup.
# os.environ["SPARK_SUBMIT_OPTS"] = (
#     "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
#     "--add-opens=java.base/java.nio=ALL-UNNAMED"
# )

print("JAVA_HOME =", java_home)
!java -version

In [ ]:
import pydeequ
from pyspark.sql import SparkSession, Row

spark = (
    SparkSession.builder
    .config("spark.jars.packages", pydeequ.deequ_maven_coord)   # pulls matching Deequ JAR from Maven
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord)     # avoids a known dependency clash
    .config("spark.sql.shuffle.partitions", "4")                # small demo data; keep shuffles cheap
    .getOrCreate()
)

print("Spark:", spark.version)
print("Deequ coordinate:", pydeequ.deequ_maven_coord)

## 1. A small dataset with deliberate problems

We use a tiny synthetic product table so you can verify the results by eye. Everything here runs identically on billions of rows — that's the whole point of doing this on Spark.

Look closely: there are four bugs baked in.

In [ ]:
data = [
    Row(id=1, product="Widget Pro",   priority="high", num_views=100, price=29.99,  country="US"),
    Row(id=2, product="Widget Lite",  priority="low",  num_views=40,  price=9.99,   country="DE"),
    Row(id=3, product="Gadget Max",   priority="high", num_views=0,   price=-5.00,  country="FR"),
    Row(id=4, product=None,           priority="low",  num_views=12,  price=14.50,  country="US"),
    Row(id=4, product="Gizmo Mini",   priority="urgent", num_views=7, price=4.25,   country="JP"),
]

df = spark.createDataFrame(data)
df.show(truncate=False)

## 2. The core: `Check` + `VerificationSuite`

This is the heart of Deequ. A **`Check`** is a named group of **constraints**. A constraint is an assertion about the data — *"this column is complete"*, *"this column is unique"*, *"values fall in this set"*.

A check has a **level**: `CheckLevel.Error` (a failure should block the pipeline) or `CheckLevel.Warning` (worth flagging, not fatal).

The **`VerificationSuite`** runs the check(s) against a DataFrame and returns results. Crucially, Deequ computes the underlying metrics in **one optimized pass** over the data rather than scanning once per constraint.

In [ ]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult

check = (
    Check(spark, CheckLevel.Error, "Product table quality checks")
    .hasSize(lambda sz: sz >= 3)                      # at least 3 rows
    .isComplete("id")                                 # no nulls in id
    .isUnique("id")                                   # id is a primary key
    .isComplete("product")                            # every product named
    .isContainedIn("priority", ["high", "low"])       # allowed values only
    .isNonNegative("price")                           # price >= 0
    .isComplete("country")
)

result = (
    VerificationSuite(spark)
    .onData(df)
    .addCheck(check)
    .run()
)

result_df = VerificationResult.checkResultsAsDataFrame(spark, result)
result_df.select("constraint", "constraint_status", "constraint_message").show(truncate=False)

Read the `constraint_status` column. You should see **`Success`** for the constraints the data satisfies and **`Failure`** for the four planted bugs, each with a `constraint_message` explaining what went wrong.

This is the moment the concept lands: *we described what good data looks like, and Deequ told us precisely where reality disagreed.*

In [ ]:
# Filter to just the failures — this is what you'd surface in a pipeline alert.
from pyspark.sql.functions import col

(result_df
 .select("constraint", "constraint_status", "constraint_message")
 .filter(col("constraint_status") == "Failure")
 .show(truncate=False))

### 2a. Fix the data and make it pass

**Fix the data and re-run the exact same check.**

Watch every row flip to `Success`. The check didn't change — the data got better. That's the contract Deequ enforces.

In [ ]:
clean_data = [
    Row(id=1, product="Widget Pro",  priority="high", num_views=100, price=29.99, country="US"),
    Row(id=2, product="Widget Lite", priority="low",  num_views=40,  price=9.99,  country="DE"),
    Row(id=3, product="Gadget Max",  priority="high", num_views=0,   price=19.00, country="FR"),
    Row(id=4, product="Gizmo Mini",  priority="low",  num_views=7,   price=4.25,  country="JP"),
]
clean_df = spark.createDataFrame(clean_data)

clean_result = VerificationSuite(spark).onData(clean_df).addCheck(check).run()
VerificationResult.checkResultsAsDataFrame(spark, clean_result) \
    .select("constraint", "constraint_status").show(truncate=False)

## 3. Error vs Warning, and richer constraints

Real checks mix *must-not-ship* rules with *keep-an-eye-on-this* rules. Below, completeness/uniqueness are **Errors**, while a softer distribution expectation is a **Warning**.

Notice constraints can take a **lambda** describing the acceptable range of a metric — e.g. *"at least 90% of rows are complete"* rather than a hard 100%. This is how you encode realistic tolerances.

In [ ]:
errors = (
    Check(spark, CheckLevel.Error, "Must pass")
    .isComplete("id")
    .isUnique("id")
    .isNonNegative("price")
)

warnings = (
    Check(spark, CheckLevel.Warning, "Worth watching")
    .hasCompleteness("product", lambda c: c >= 0.9)        # >=90% of products named
    .hasMin("price", lambda m: m >= 0)
    .hasMax("num_views", lambda m: m <= 10000)             # sanity ceiling
    .isContainedIn("country", ["US", "DE", "FR", "JP", "GB"])
)

multi = (
    VerificationSuite(spark)
    .onData(df)
    .addCheck(errors)
    .addCheck(warnings)
    .run()
)

VerificationResult.checkResultsAsDataFrame(spark, multi) \
    .select("check", "check_level", "check_status", "constraint_status", "constraint") \
    .show(truncate=False)

The `check_status` aggregates the constraints within each check. An `Error`-level check that has any failing constraint reports `Error`; the `Warning`-level check reports `Warning`. In a pipeline you typically **abort on `Error`** and **log on `Warning`**.

## 4. Under the hood: Analyzers (metrics without pass/fail)

Every constraint is built on an **Analyzer** that computes a numeric *metric* (completeness = 0.8, distinctness = 1.0, mean price = 10.5, …). Sometimes you just want the raw numbers — for a data-profiling dashboard, or to decide what thresholds to set in the first place. `AnalysisRunner` gives you exactly that, decoupled from any pass/fail logic.

In [ ]:
from pydeequ.analyzers import AnalysisRunner, AnalyzerContext, \
    Size, Completeness, Distinctness, Mean, Maximum, ApproxCountDistinct

analysis = (
    AnalysisRunner(spark)
    .onData(df)
    .addAnalyzer(Size())
    .addAnalyzer(Completeness("product"))
    .addAnalyzer(Distinctness("id"))
    .addAnalyzer(Mean("price"))
    .addAnalyzer(Maximum("num_views"))
    .addAnalyzer(ApproxCountDistinct("country"))
    .run()
)

AnalyzerContext.successMetricsAsDataFrame(spark, analysis).show(truncate=False)

These metrics are the same values the constraints in Section 2–3 evaluated internally. Seeing them explicitly demystifies what a constraint actually *is*: **a metric plus an assertion on that metric.**

## 5. Teaser: let Deequ suggest the constraints

Writing every rule by hand doesn't scale to wide tables. **Constraint Suggestion** profiles each column and proposes constraints automatically — completeness, type, value ranges, allowed categories — which you can then review, keep, or tighten.

Deequ can *bootstrap* your data tests, and you refine from there.

In [ ]:
from pydeequ.suggestions import ConstraintSuggestionRunner, DEFAULT

suggestions = (
    ConstraintSuggestionRunner(spark)
    .onData(df)
    .addConstraintRule(DEFAULT())
    .run()
)

for s in suggestions["constraint_suggestions"]:
    print(f"[{s['column_name']}] {s['description']}")
    print(f"    -> code: {s['code_for_constraint']}\n")

Each suggestion includes a human description **and** the exact PyDeequ code (`code_for_constraint`) you'd paste into a `Check`. The intended workflow: run suggestion once on a trusted sample, copy the constraints you agree with into a hand-curated check, and version-control that check alongside your pipeline.

## 6. Building a real pipeline

Let's assemble everything into a small but realistic batch job with the shape you'd actually deploy. The stages are:

1. **Ingest** — read raw records (here we simulate reading a daily drop).
2. **Validate** — run the Deequ gate. *This is the guardrail.* An `Error` aborts the whole job before any bad data is written.
3. **Transform** — only runs if validation passed: clean/enrich the data.
4. **Write** — persist the curated output for downstream consumers.

The key property: stages 3 and 4 are **unreachable** unless stage 2 passes. That's what makes Deequ a gate rather than a report.

> **This section is self-contained.** It redefines its own checks, data, and helper so you can run it on its own — the only prerequisites are the `spark` session and imports from **Section 0**. (It intentionally duplicates a few definitions from earlier sections for that reason.)

In [ ]:
import tempfile, os
from pyspark.sql import Row
from pyspark.sql.functions import col, upper, round as spark_round
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult


# ---- Self-contained definitions for this section --------------------------

class DataQualityError(Exception):
    """Raised when an Error-level data quality check fails — aborts the pipeline."""
    pass


def build_checks():
    """The data contract: hard Errors that must pass, soft Warnings worth flagging."""
    errors = (
        Check(spark, CheckLevel.Error, "Must pass")
        .isComplete("id")
        .isUnique("id")
        .isNonNegative("price")
    )
    warnings = (
        Check(spark, CheckLevel.Warning, "Worth watching")
        .hasCompleteness("product", lambda c: c >= 0.9)
        .isContainedIn("country", ["US", "DE", "FR", "JP", "GB"])
    )
    return errors, warnings


def gate(verification_result, stage_name):
    """Branch on overall status: raise on Error, warn on Warning, else proceed."""
    status = verification_result.status                      # "Success" | "Warning" | "Error"
    rows = VerificationResult.checkResultsAsDataFrame(spark, verification_result)
    failures = [r for r in rows.collect() if r["constraint_status"] == "Failure"]

    if status == "Success":
        print(f"[{stage_name}] PASSED — all checks green.")
        return failures

    detail = "\n".join(
        f"    - ({r['check_level']}) {r['constraint']}: {r['constraint_message']}"
        for r in failures
    )
    if status == "Error":
        raise DataQualityError(f"[{stage_name}] ABORTED — Error-level failure:\n{detail}")

    import warnings as _w
    _w.warn(f"[{stage_name}] WARNINGS (continuing):\n{detail}")
    print(f"[{stage_name}] passed with warnings — proceeding.")
    return failures


# ---- Self-contained input data (two simulated daily drops) ----------------

dirty_drop = spark.createDataFrame([
    Row(id=1, product="Widget Pro",  priority="high", num_views=100, price=29.99, country="US"),
    Row(id=3, product="Gadget Max",  priority="high", num_views=0,   price=-5.00, country="FR"),  # negative price
    Row(id=4, product=None,          priority="low",  num_views=12,  price=14.50, country="US"),  # null product
    Row(id=4, product="Gizmo Mini",  priority="low",  num_views=7,   price=4.25,  country="JP"),  # dup id
])

clean_drop = spark.createDataFrame([
    Row(id=1, product="Widget Pro",  priority="high", num_views=100, price=29.99, country="US"),
    Row(id=2, product="Widget Lite", priority="low",  num_views=40,  price=9.99,  country="DE"),
    Row(id=3, product="Gadget Max",  priority="high", num_views=0,   price=19.00, country="FR"),
    Row(id=4, product="Gizmo Mini",  priority="low",  num_views=7,   price=4.25,  country="JP"),
])

# A writable scratch dir for this demo's "data lake" (Colab-friendly /tmp).
LAKE = tempfile.mkdtemp(prefix="deequ_lake_")
print("Demo lake at:", LAKE)

In [ ]:
# ---- The four pipeline stages ---------------------------------------------

def ingest(raw_df):
    """In production this reads from S3/GCS/a table. Here we pass a DataFrame in."""
    print("[ingest] reading raw records ...")
    return raw_df


def validate(raw_df, stage_name):
    """Run the Deequ gate. Raises DataQualityError on Error-level failure."""
    print("[validate] running data quality checks ...")
    errors, warnings = build_checks()
    res = (
        VerificationSuite(spark)
        .onData(raw_df)
        .addCheck(errors)
        .addCheck(warnings)
        .run()
    )
    gate(res, stage_name)
    return res


def transform(raw_df):
    """Only reached if validation passed. Curate the data for consumers."""
    print("[transform] cleaning and enriching ...")
    return (
        raw_df
        .withColumn("country", upper(col("country")))
        .withColumn("price", spark_round(col("price"), 2))
        .filter(col("num_views") >= 0)
    )


def write(curated_df, name):
    """Persist curated output. Parquet here; a managed table in production."""
    out = os.path.join(LAKE, name)
    print(f"[write] writing curated output -> {out}")
    curated_df.write.mode("overwrite").parquet(out)
    return out


def daily_job(raw_df, run_label):
    """Orchestrate the stages. transform/write are unreachable unless validate passes."""
    print(f"\n========== DAILY JOB: {run_label} ==========")
    try:
        raw = ingest(raw_df)
        validate(raw, stage_name=f"{run_label}:validate")     # <-- the gate
        curated = transform(raw)                              # only if gate passed
        path = write(curated, name=run_label)                 # only if gate passed
        print(f"JOB SUCCEEDED. Curated data at: {path}")
        return path
    except DataQualityError as e:
        # In Airflow/Databricks you would NOT catch this — letting it propagate
        # fails the task and blocks downstream tasks. We catch only so the demo
        # can show both outcomes in one notebook.
        print("JOB FAILED at validation. No data written, downstream skipped.")
        print(e)
        return None

Let's run the job twice on two different "daily drops" to see the gate do its work.

In [ ]:
# Day 1: a bad drop -> Error-level failure -> job aborts before writing anything.
day1 = daily_job(dirty_drop, run_label="2026-05-21")

# Day 2: a good drop -> passes the gate -> transform + write run.
day2 = daily_job(clean_drop, run_label="2026-05-22")

In [ ]:
# Prove the write only happened for the good run: read the curated output back.
if day2:
    print("Reading back the curated table written by the successful run:\n")
    spark.read.parquet(day2).show(truncate=False)

Notice:

* the **bad drop never produced an output file** — the job stopped at validation and the `transform`/`write` stages were never called
* the **good drop flowed all the way through** to a curated Parquet table. The only thing that decided the difference was data quality.

## Wrap-up: where this fits and why it matters

**Summary:**
- Wrap the `VerificationSuite` in your job and **raise on any `Error`-level failure**, so the orchestrator (Airflow, Databricks Workflows, etc.) marks the run failed and halts downstream steps — exactly what Section 6's `daily_job` demonstrated.
- Persist metrics with a **`MetricsRepository`** to track quality trends and add **anomaly detection** over time.
- Treat your checks like test code: **review them, version them, and evolve them** as the data contract changes.

**Takeaway:**
* *Deequ turns "I think the data looks fine" into an assertion the machine can verify every single run.*

In [ ]:
# Clean up
spark.stop()